In [16]:
import lightgbm as lgb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, mean_squared_error, r2_score
from sklearn.datasets import make_classification, make_regression

import warnings
warnings.filterwarnings("ignore")

# Regression

In [34]:
def reg_data():
    X, y = make_regression(
        n_samples=10000,    # 10,000 houses
        n_features=20,      # 20 features (size, bedrooms, location, etc.)
        n_informative=15,   # 15 actually useful features
        noise=10,           # Some random noise
        random_state=42
    )
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )
    return X_train, X_test, y_train, y_test

In [3]:
X_train

array([[-0.77077681,  1.29779838, -0.11421044, ...,  1.55167538,
         0.60289472, -0.10986175],
       [ 1.23822642, -0.24402657, -1.35874034, ..., -2.42490072,
        -0.18208393,  0.07027384],
       [ 1.27209138,  0.78355808, -0.66427101, ...,  0.78868454,
         1.30334499, -0.12504555],
       ...,
       [ 0.08042462,  0.93267768,  1.73674717, ...,  0.08799113,
         0.72055958,  1.59771556],
       [ 0.3875834 ,  0.66152363, -0.81117835, ...,  1.11076048,
         1.16962509,  0.58679708],
       [-0.48348391, -0.81793729,  0.8609101 , ..., -0.95508229,
         0.1405089 , -1.68206728]], shape=(8000, 20))

In [4]:
X_train.shape,X_test.shape

((8000, 20), (2000, 20))

In [5]:
y_train.shape

(8000,)

In [6]:
model = lgb.LGBMRegressor(
    num_leaves = 31,
    max_depth = -1,
    learning_rate=0.1,
    n_estimators=100,
    objective='regression',
    lambda_l1=0.0,
    lambda_l2=0.0,
    min_gain_to_split=0.0,
    boosting_type='gbdt', #   'gbdt' = traditional gradient boosting (default, most accurate)
    metric='rmse',
    random_state = 42,
    n_jobs = -1,
    verbose=-1, # -1 = silent
)

In [8]:
model.get_params()

{'boosting_type': 'gbdt',
 'class_weight': None,
 'colsample_bytree': 1.0,
 'importance_type': 'split',
 'learning_rate': 0.1,
 'max_depth': -1,
 'min_child_samples': 20,
 'min_child_weight': 0.001,
 'min_split_gain': 0.0,
 'n_estimators': 100,
 'n_jobs': -1,
 'num_leaves': 31,
 'objective': 'regression',
 'random_state': 42,
 'reg_alpha': 0.0,
 'reg_lambda': 0.0,
 'subsample': 1.0,
 'subsample_for_bin': 200000,
 'subsample_freq': 0,
 'lambda_l1': 0.0,
 'lambda_l2': 0.0,
 'min_gain_to_split': 0.0,
 'metric': 'rmse',
 'verbose': -1}

In [12]:
model.fit(X_train,y_train)

,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.1
,n_estimators,100
,subsample_for_bin,200000
,objective,'regression'
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [17]:
y_pred = model.predict(X_test)

In [15]:
y_test[:5],y_pred[:5]

(array([ -55.85637203,   53.48454116, -415.3538054 ,  433.19357559,
        -369.20633395]),
 array([   8.70655646,   24.57705653, -349.84312104,  387.60053495,
        -280.48316311]))

In [18]:
train_pred = model.predict(X_train)
train_rmse = np.sqrt(mean_squared_error(y_train, train_pred))
test_rmse = np.sqrt(mean_squared_error(y_test, y_pred))

train_r2 = r2_score(y_train, train_pred)
test_r2 = r2_score(y_test, y_pred)

print("Model Performance:")
print(f"Training RMSE: {train_rmse:.3f}")
print(f"Test RMSE: {test_rmse:.3f}")
print(f"Training R²: {train_r2:.3f}")
print(f"Test R²: {test_r2:.3f}")

Model Performance:
Training RMSE: 27.613
Test RMSE: 56.442
Training R²: 0.988
Test R²: 0.949


In [17]:
# INTERPRETATION:
# If train_rmse=10 and test_rmse=12:
#   - Good! Small gap means not overfitting much
#
# If train_rmse=5 and test_rmse=25:
#   - Overfitting! Solutions:
#     * Decrease num_leaves (31 → 15)
#     * Increase min_data_in_leaf (20 → 100)
#     * Increase lambda_l2 (0 → 10)
#     * Set max_depth (e.g., 7)
#
# If train_rmse=20 and test_rmse=22:
#   - Underfitting! Solutions:
#     * Increase num_leaves (31 → 63)
#     * Increase n_estimators (100 → 500)
#     * Decrease min_data_in_leaf (20 → 5)

In [19]:
# Our model is undefitting 

In [20]:
model = lgb.LGBMRegressor(
    num_leaves = 63,
    max_depth = -1,
    learning_rate=0.1,
    n_estimators=500,
    objective='regression',
    lambda_l1=0.0,
    lambda_l2=0.0,
    min_data_in_leaf=5,
    min_gain_to_split=0.0,
    boosting_type='gbdt', #   'gbdt' = traditional gradient boosting (default, most accurate)
    metric='rmse',
    random_state = 42,
    n_jobs = -1,
    verbose=-1, # -1 = silent
)

In [21]:
model.fit(X_train,y_train)

,boosting_type,'gbdt'
,num_leaves,63
,max_depth,-1
,learning_rate,0.1
,n_estimators,500
,subsample_for_bin,200000
,objective,'regression'
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [22]:
y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

train_rmse = np.sqrt(mean_squared_error(y_train,y_pred_train))
test_rmse = np.sqrt(mean_squared_error(y_test,y_pred_test))

print(f"Train RMSE: {train_rmse}")
print(f"Test RMSE: {test_rmse}")

Train RMSE: 3.12822818903543
Test RMSE: 57.41918935514325


In [23]:
# our model is overfitting, will do hyperparameter tunning 

# Classification

In [24]:
X, y = make_classification(
    n_samples=10000,
    n_features=30,
    n_informative=20,
    n_redundant=5,
    n_classes=2,  # Binary classification
    random_state=42
)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [27]:
classifier = lgb.LGBMClassifier(
    boosting_type='gbdt',
    num_leaves = 31,
    max_depth=-1,
    learning_rate=0.1,
    n_estimators=100,
    objective = 'binary',
    metric='binary_logloss',
    is_unbalance = False,
    bagging_fraction = 0.8,
    feature_fraction=0.8,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

In [28]:
classifier.get_params()

{'boosting_type': 'gbdt',
 'class_weight': None,
 'colsample_bytree': 1.0,
 'importance_type': 'split',
 'learning_rate': 0.1,
 'max_depth': -1,
 'min_child_samples': 20,
 'min_child_weight': 0.001,
 'min_split_gain': 0.0,
 'n_estimators': 100,
 'n_jobs': -1,
 'num_leaves': 31,
 'objective': 'binary',
 'random_state': 42,
 'reg_alpha': 0.0,
 'reg_lambda': 0.0,
 'subsample': 1.0,
 'subsample_for_bin': 200000,
 'subsample_freq': 0,
 'metric': 'binary_logloss',
 'is_unbalance': False,
 'bagging_fraction': 0.8,
 'feature_fraction': 0.8,
 'verbose': -1}

In [29]:
classifier.fit(X_train,y_train)

,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.1
,n_estimators,100
,subsample_for_bin,200000
,objective,'binary'
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [30]:
y_pred_test = classifier.predict(X_test)
y_pred_proba_test = classifier.predict_proba(X_test)
print(y_test[:5])
print(y_pred_test[:5])
print(f"accuracy: {accuracy_score(y_test,y_pred_test)}")

[0 1 0 0 1]
[0 1 0 0 1]
accuracy: 0.9585


In [31]:
from sklearn.metrics import classification_report, confusion_matrix

In [33]:
cm = confusion_matrix(y_test,y_pred_test)
print("confusion matric:\n",cm)
print("classification_report:")
print(classification_report(y_test,y_pred_test))

confusion matric:
 [[933  35]
 [ 48 984]]
classification_report:
              precision    recall  f1-score   support

           0       0.95      0.96      0.96       968
           1       0.97      0.95      0.96      1032

    accuracy                           0.96      2000
   macro avg       0.96      0.96      0.96      2000
weighted avg       0.96      0.96      0.96      2000



# LIGHTGBM NATIVE API


In [35]:
X_train, X_test, y_train, y_test = reg_data()

In [40]:
[f"feature_{i}" for i in range(X_train.shape[1])]

['feature_0',
 'feature_1',
 'feature_2',
 'feature_3',
 'feature_4',
 'feature_5',
 'feature_6',
 'feature_7',
 'feature_8',
 'feature_9',
 'feature_10',
 'feature_11',
 'feature_12',
 'feature_13',
 'feature_14',
 'feature_15',
 'feature_16',
 'feature_17',
 'feature_18',
 'feature_19']

In [42]:
train_data = lgb.Dataset(
    data=X_train,
    label=y_train,
    feature_name = [f"feature_{i}" for i in range(X_train.shape[1])],
    categorical_feature = 'auto',# Auto-detect categorical features
                                 # Or specify: [0, 2, 5] (column indices)
                                 # THEORY: LightGBM handles categoricals natively!
    free_raw_data=False  # Keep raw data for later use
)
test_data = lgb.Dataset(
    data=X_test,
    label = y_test,
    reference=train_data,# Use same binning as training data
    free_raw_data=False
)

In [43]:
# Parameters as dictionary
params = {
    # Task parameters
    'objective': 'regression',
    'metric': ['rmse', 'mae'],  # Can track multiple metrics!
    'boosting_type': 'gbdt',
    
    # Tree parameters
    'num_leaves': 31,
    'max_depth': -1,
    'min_data_in_leaf': 20,
    
    # Learning parameters
    'learning_rate': 0.1,
    
    # Sampling parameters
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'feature_fraction': 0.8,
    
    # Regularization
    'lambda_l1': 0.0,
    'lambda_l2': 0.0,
    
    # Other
    'verbose': -1,
    'seed': 42
}

In [44]:
model_native = lgb.train(
    params=params,
    train_set = train_data,
    num_boost_round = 100,
    valid_sets=[train_data,test_data], # Datasets to monitor
    valid_names = ["train","test"], # Names for monitoring
    callbacks=[lgb.early_stopping(stopping_rounds=10)] # List of callback functions that are applied at each iteration.
)

Training until validation scores don't improve for 10 rounds
Did not meet early stopping. Best iteration is:
[100]	train's rmse: 29.2893	train's l1: 23.1607	test's rmse: 53.6494	test's l1: 41.22


In [46]:
y_pred_native = model_native.predict(X_test,num_iteration=model_native.best_iteration) # Use best iteration (if early stopping)

In [47]:
test_rmse = np.sqrt(mean_squared_error(y_test,y_pred_native))
test_r2 = r2_score(y_test,y_pred_native)

print(f"Rmse:{test_rmse}")
print(f"r2 score:{test_r2}")

Rmse:53.64939413549829
r2 score:0.9538755228281386


# CROSS-VALIDATION

In [48]:
# create full dataset
X, y = make_regression(
        n_samples=10000,    # 10,000 houses
        n_features=20,      # 20 features (size, bedrooms, location, etc.)
        n_informative=15,   # 15 actually useful features
        noise=10,           # Some random noise
        random_state=42
)
full_data = lgb.Dataset(X,label=y)

In [50]:
# Parameters as dictionary
params = {
    # Task parameters
    'objective': 'regression',
    'metric': ['rmse', 'mae'],  # Can track multiple metrics!
    'boosting_type': 'gbdt',
    
    # Tree parameters
    'num_leaves': 31,
    'max_depth': -1,
    'min_data_in_leaf': 20,
    
    # Learning parameters
    'learning_rate': 0.1,
    
    # Sampling parameters
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'feature_fraction': 0.8,
    
    # Regularization
    'lambda_l1': 0.0,
    'lambda_l2': 0.0,
    
    # Other
    'verbose': -1,
    'seed': 42
}

In [52]:
cv_result = lgb.cv(
    params,
    train_set=full_data,
    num_boost_round=100,
    nfold=5, # # Trains 5 models, each using 4 parts for training, 1 for validation
    stratified=False,  # Key: Uses regular KFold for regression, True for classification
    seed=42,
    return_cvbooster=False, # If True, returns all 5 trained models
)

In [54]:
pd.DataFrame(cv_result)

,valid rmse-mean,valid rmse-stdv,valid l1-mean,valid l1-stdv
0,239.551978,1.468557,190.982725,2.291120
1,230.320926,1.512083,183.467919,2.206163
2,222.741472,1.433066,177.359500,2.110369
3,215.375684,1.379762,171.387879,1.971071
4,208.752688,1.353847,166.082639,1.869086
...,...,...,...,...
95,54.236825,0.753755,41.683769,0.641199
96,53.938141,0.761366,41.472084,0.636392
97,53.683366,0.761778,41.282711,0.636929
98,53.422987,0.746874,41.090557,0.626400


In [55]:
for key in cv_result:
    print(f"{key}: {cv_result[key][-5:]}")

valid rmse-mean: [54.236825418586115, 53.938140885409425, 53.68336561150802, 53.422986762946735, 53.18255910258658]
valid rmse-stdv: [0.7537551918295372, 0.7613659127244221, 0.7617780391386477, 0.7468740236564994, 0.7679192525514991]
valid l1-mean: [41.68376915667044, 41.47208428551001, 41.282710837557175, 41.09055737874634, 40.924816349424546]
valid l1-stdv: [0.6411992711859076, 0.6363918688724344, 0.6369290463661536, 0.6264001805191358, 0.6409357559631298]


In [58]:
# best results
best_iteration = len(cv_result["valid rmse-mean"])
best_rmse = cv_result["valid rmse-mean"][-1]
best_stdv = cv_result["valid rmse-stdv"][-1]

print(f"Best RMSE: {best_rmse}  (+- {best_stdv})")

Best RMSE: 53.18255910258658  (+- 0.7679192525514991)


INTERPRETATION:
If RMSE = 45.2 ± 2.1:
  - Average error is 45.2 across 5 folds
  - Standard deviation is 2.1 (low = consistent performance)

If RMSE = 45.2 ± 15.3:
  - Average is 45.2 but varies a lot
  - High std = unstable, performance depends on data split
  - Might indicate overfitting or data quality issues

#### GPU training
For GPU training, just add 'device': 'gpu' to parameters in native api \
device='gpu' in sklearn api